### RAG Pipelines - Data Ingesion to 

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/rt/wd9f2l9527b94k0k8xjwhrym0000gn/T/ipykernel_32301/1573117932.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/sujata/Documents/Job Prep/YTRAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory



def process_all_pdfs(pdf_directory):
    """ Process all PDF files in a directory """
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information etadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

## Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")




Found 2 PDF files to process

 Processing: Denoising_Report.pdf
 Loaded 9 pages

 Processing: Floe Sparse 4DGS.pdf
 Loaded 10 pages

 Total documents loaded: 19


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-13T14:41:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-13T14:41:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdfs/Denoising_Report.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Denoising_Report.pdf', 'file_type': 'pdf'}, page_content='Self-Supervised Image Denoising:\nA Comparative Study of Single-Image\nMethods\nSujata Chaudhury\nApril 13, 2026\nAbstract\nImage denoising is a fundamental problem in low-level computer vision, requiring\nthe recovery of a clean image from its noisy observation. Supervised deep learning\nmethods achieve state-of-the-art performance but require large paired or unpaired\ndatasets. In this report, we evaluate three self-supervised and unsupervised denoising\nmethods — Dee

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]  
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)

chunks

Split 19 documents into 93 chunks

Example chunk:
Content: Self-Supervised Image Denoising:
A Comparative Study of Single-Image
Methods
Sujata Chaudhury
April 13, 2026
Abstract
Image denoising is a fundamental problem in low-level computer vision, requiring
t...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-13T14:41:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-13T14:41:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdfs/Denoising_Report.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Denoising_Report.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-13T14:41:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-13T14:41:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdfs/Denoising_Report.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Denoising_Report.pdf', 'file_type': 'pdf'}, page_content='Self-Supervised Image Denoising:\nA Comparative Study of Single-Image\nMethods\nSujata Chaudhury\nApril 13, 2026\nAbstract\nImage denoising is a fundamental problem in low-level computer vision, requiring\nthe recovery of a clean image from its noisy observation. Supervised deep learning\nmethods achieve state-of-the-art performance but require large paired or unpaired\ndatasets. In this report, we evaluate three self-supervised and unsupervised denoising\nmethods — Dee

### Embedding And VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """ Handles document embedding generation using """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manger

        Args:
        model_name: HuggingFace model name for sentence embeddings

        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Load the SentenceTransformer model """

        try:
            print(f"Loadingembedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loadingembedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8263.61it/s]


Model loaded successfully. Embedding dimension : 384


### VectorStore

In [8]:
class VectorStore:
    def __init__ (self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 65


In [9]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['Self-Supervised Image Denoising:\nA Comparative Study of Single-Image\nMethods\nSujata Chaudhury\nApril 13, 2026\nAbstract\nImage denoising is a fundamental problem in low-level computer vision, requiring\nthe recovery of a clean image from its noisy observation. Supervised deep learning\nmethods achieve state-of-the-art performance but require large paired or unpaired\ndatasets. In this report, we evaluate three self-supervised and unsupervised denoising\nmethods — Deep Image Prior (DIP), Self2Self, and Neighbor2Neighbor — on 16\nimages from the CBSD68 benchmark under additive white Gaussian noise with\nσ = 25. Each method is evaluated in a single-image setting, and Neighbor2Neighbor\nis additionally evaluated in a pretrained setting to establish an upper-bound reference.\nResults are reported in terms of Peak Signal-to-Noise Ratio (PSNR), Structural\nSimilarity Index (SSIM), edge preservation, and inference time. The pretrained',
 'Results are reported in terms of Peak Signal-to-No

In [10]:
texts = [doc.page_content for doc in chunks]

### Ggenerate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 93 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  4.50it/s]

Generated embeddings with shape: (93, 384)
Adding 93 documents to vector store...
Successfully added 93 documents to vector store
Total documents in collection: 158


### Retriever Pipeline from VectoreStore

In [12]:
class RAGRetriever:
    def __init__ (self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [13]:
rag_retriever.retrieve("Sparse4DGS")

Retrieving documents for query: 'Sparse4DGS'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_8e63e5a4_13',
  'content': 'and lack the optimization of geometric structures and inter-frame\ncontinuity. In contrast, our Sparse4DGS employs a novel global-\nlocal feature encoding approach to focus on local regions, which\ncan better represent sparse scenarios. Moreover, by integrating the\nflow MLP and depth assistance, it guides the alignment of geomet-\nric structures and compensates for inter-frame features, thereby\nachieving a more effective representation of sparse dynamic scenes\nand higher rendering quality.\n10643',
  'metadata': {'title': 'Sparse4DGS: Flow-Geometry Assisted 4D Gaussian Splatting for Dynamic Sparse View Synthesis',
   'source': '../data/pdfs/Floe Sparse 4DGS.pdf',
   'keywords': '3D Gaussian Splatting; Sparse-View Synthesis; Dynamic Scene Reconstruction',
   'creator': 'LaTeX with acmart 2025/05/30 v2.14 Typesetting articles for the Association for Computing Machinery and hyperref 2024-01-20 v7.01h Hypertext links for LaTeX',
   'page_label': 

### Integration Vectordb Context pipeline With LLM output

In [19]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key = groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

## 2. SImple RAG function: retrieve context = generate response

def rag_simple(query, retrieve, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=""" Use the following context to answer the question precisely.
    Context:
    {context}

    Question: {query}

    Answer: """

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

### Enhanced RAG Pipeline Features

In [22]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is denoising", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is denoising'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer: Denoising is the process of removing unwanted noise from an image by restoring each pixel (or patch) using information from other parts of the image—often by averaging or transforming groups of similar patches—to recover the underlying clean signal.
Sources: [{'source': 'Denoising_Report.pdf', 'page': 1, 'score': 0.10018110275268555, 'preview': 'denoising methods based on non-local self-similarity. Each pixel is restored as a weighted\naverage of all pixels in the image whose local neighbourhoods are similar, exploiting\nthe recurring patch structure of natural images. While effective on textures, NLM is\ncomputationally expensive and tends to...'}]
Confidence: 0.10018110275268555
Context Preview: denoising methods based on non-local self-similarity. Each pixel is restored as a weighted
average of all pixels in the image whose local neighbourhoods are similar, exploiting
the recurring patch struct

### Agentic RAG Pipeline

### State Definition

AttributeError: 'numpy.ndarray' object has no attribute 'embed_documents'